# 09. Winner-Locked Candidate Pools and the Stage-1-to-Stage-2 Interface — Herbal Supplements

This notebook resolves the Notebook 07 query-only winner and the Notebook 08 personalized winner from their machine-readable contracts. It does not rerun retrieval, compare alternative methods, or select a new winner.

Exactly two top-1,000 candidate pools are materialized in both long and list formats: the Dense–Sparse Hybrid query-only pool and the Profile Sparse QCHS personalized pool. Candidate identities, ordering, and scores are copied from their respective upstream winner artifacts. Target indicators and candidate Brand fields are verified against item identities and the Notebook 04 catalog without changing the upstream order.

The export validates complete case coverage, exact candidate counts, unique items within each case, contiguous ranks, finite non-increasing scores, catalog membership, Brand consistency, and unchanged regime counts. For every QCHS fallback case, the query-only and personalized pools must match exactly in candidate identity, rank, score, and target indicator.

The stored execution exports 1,968,000 rows per pool, or 3,936,000 candidate rows in total, and preserves 1,415 exact fallback cases. It also writes descriptive Stage-1 exposure and rank summaries. These summaries do not alter the frozen pools or introduce a new selection decision.

The two locked pools form the immutable Stage-1-to-Stage-2 interface used by the reranking and aggregate-analysis notebooks.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%pip install -q pyarrow

In [3]:
# ==== Load Libraries ====
from pathlib import Path
import json

import numpy as np
import pandas as pd

In [4]:
# ==== Define Winner Contracts, Pool Paths, and Outputs ====
CATEGORY_ID = 'herbal'
CATEGORY_FOLDER = 'herbal_supplements'
CATEGORY_LABEL = 'Herbal Supplements'

PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories") / CATEGORY_FOLDER

QUERY_CACHE_PATH = PROJECT_ROOT / 'outputs/query_cache/herbal_query_cache.parquet'
ITEM_DOCS_PATH = PROJECT_ROOT / 'data/processed/items/item_docs_herbal.parquet'
STAGE1_WINNER_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs/stage1_query_retrieval_selection"
    / f"stage1_query_only_winner_{CATEGORY_ID}.json"
)
PERSONALIZED_RUN_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs/stage1_personalized_retrieval"
    / f"personalized_retrieval_manifest_{CATEGORY_ID}.json"
)
PERSONALIZED_WINNER_MANIFEST_PATH = (
    PROJECT_ROOT
    / "outputs/stage1_personalized_retrieval"
    / f"personalized_retrieval_winner_{CATEGORY_ID}.json"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs/stage1_candidate_pools"
BY_METHOD_DIR = OUTPUT_DIR / "by_method"

QUERY_ONLY_WINNER_POOL_PATH = BY_METHOD_DIR / f"query_only_winner_top1000_{CATEGORY_ID}.parquet"
QUERY_ONLY_WINNER_LIST_PATH = BY_METHOD_DIR / f"query_only_winner_top1000_list_{CATEGORY_ID}.parquet"
PERSONALIZED_WINNER_POOL_PATH = BY_METHOD_DIR / f"personalized_winner_top1000_{CATEGORY_ID}.parquet"
PERSONALIZED_WINNER_LIST_PATH = BY_METHOD_DIR / f"personalized_winner_top1000_list_{CATEGORY_ID}.parquet"

SUMMARY_PATH = OUTPUT_DIR / f"winner_candidate_pool_summary_{CATEGORY_ID}.csv"
QC_PATH = OUTPUT_DIR / f"candidate_pool_export_qc_{CATEGORY_ID}.csv"
COLD_CANDIDATE_FALLBACK_QC_PATH = OUTPUT_DIR / f"cold_candidate_fallback_qc_{CATEGORY_ID}.csv"
STAGE1_PERSONALIZATION_UPLIFT_OVERALL_PATH = OUTPUT_DIR / f"stage1_personalization_uplift_overall_{CATEGORY_ID}.csv"
STAGE1_PERSONALIZATION_UPLIFT_BY_REGIME_PATH = OUTPUT_DIR / f"stage1_personalization_uplift_by_regime_{CATEGORY_ID}.csv"
MANIFEST_PATH = OUTPUT_DIR / f"stage1_candidate_pool_export_manifest_{CATEGORY_ID}.json"

BY_METHOD_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", STAGE1_WINNER_MANIFEST_PATH)
print("Input:", PERSONALIZED_WINNER_MANIFEST_PATH)
print("Output:", QUERY_ONLY_WINNER_POOL_PATH)
print("Output:", PERSONALIZED_WINNER_POOL_PATH)


Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_query_retrieval_selection/stage1_query_only_winner_herbal.json
Input: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_personalized_retrieval/personalized_retrieval_winner_herbal.json
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/query_only_winner_top1000_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_herbal.parquet


In [5]:
# ==== Declare the Stage-Transition Candidate-Pool Contract ====
ACTIVE_QUERY_COLUMN = "query"
LEGACY_QUERY_METHOD_LABEL = "C"
MAX_CANDIDATE_DEPTH = 1000

QUERY_ONLY_WINNER_CONTRACT_VERSION = "stage1_query_only_winner_v1"
PERSONALIZED_WINNER_CONTRACT_VERSION = "stage1_personalized_retrieval_winner_v1"

EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
QUERY_EVIDENCE_SCOPE = "target_review_safe_signals_only"

BRAND_TEXT_COLUMN = "brand_facet_text"


In [6]:
# ==== Define Pool-Shape, Ranking, and Identity Checks ====
def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} missing required columns: {missing}")


def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    return " ".join(str(value).split())


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def as_list(value):
    if isinstance(value, list):
        return value
    if isinstance(value, (tuple, np.ndarray, pd.Series)):
        return list(value)
    raise TypeError(f"Expected a list-like candidate value, found {type(value).__name__}.")


def validate_ranked_list(items, scores, item_id_set, expected_k, context):
    items = [str(value) for value in as_list(items)]
    scores = np.asarray(as_list(scores), dtype=float)
    if len(items) != expected_k:
        raise RuntimeError(f"{context} candidate count must equal exact-K={expected_k}.")
    if len(items) != len(scores):
        raise RuntimeError(f"{context} candidate item and score lengths differ.")
    if len(set(items)) != len(items):
        raise RuntimeError(f"{context} contains duplicate candidate items.")
    if not set(items).issubset(item_id_set):
        raise RuntimeError(f"{context} contains an item outside the Notebook 04 catalog.")
    if not np.isfinite(scores).all():
        raise RuntimeError(f"{context} contains a non-finite candidate score.")
    if np.any(np.diff(scores) > 1e-12):
        raise RuntimeError(f"{context} candidate scores are not non-increasing.")


def validate_long_pool(
    frame,
    expected_method_slug,
    expected_method_label,
    query_case_ids,
    item_id_set,
    expected_k,
):
    required = [
        "case_id", "method_slug", "candidate_parent_asin", "candidate_rank",
        "candidate_score", "retrieval_method", "is_target",
    ]
    require_columns(frame, required, expected_method_label)
    observed_slugs = sorted(frame["method_slug"].dropna().astype(str).unique().tolist())
    if observed_slugs != [expected_method_slug]:
        raise RuntimeError(
            f"{expected_method_label} export contains unexpected method_slug values: {observed_slugs}"
        )
    observed_labels = sorted(frame["retrieval_method"].dropna().astype(str).unique().tolist())
    if observed_labels != [expected_method_label]:
        raise RuntimeError(
            f"{expected_method_label} export contains unexpected retrieval_method values: {observed_labels}"
        )
    if set(frame["case_id"].astype(str)) != query_case_ids:
        raise RuntimeError(f"{expected_method_label} case IDs do not match the active query cache.")
    if frame.duplicated(["case_id", "candidate_parent_asin"]).any():
        raise RuntimeError(f"{expected_method_label} contains duplicate case-candidate rows.")
    if not set(frame["candidate_parent_asin"].astype(str)).issubset(item_id_set):
        raise RuntimeError(f"{expected_method_label} contains a candidate outside the catalog.")
    scores = pd.to_numeric(frame["candidate_score"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(scores).all():
        raise RuntimeError(f"{expected_method_label} contains non-finite scores.")

    rank_summary = frame.groupby("case_id", sort=False)["candidate_rank"].agg(
        rank_min="min", rank_max="max", rank_nunique="nunique", candidate_count="size"
    )
    bad_rank = rank_summary[
        ~rank_summary["rank_min"].eq(1)
        | ~rank_summary["rank_max"].eq(expected_k)
        | ~rank_summary["rank_nunique"].eq(expected_k)
        | ~rank_summary["candidate_count"].eq(expected_k)
    ]
    if not bad_rank.empty:
        raise RuntimeError(
            f"{expected_method_label} ranks must be exactly 1 through {expected_k}. "
            f"Examples: {bad_rank.head(10).reset_index().to_dict('records')}"
        )

    sorted_pool = frame.sort_values(["case_id", "candidate_rank"], kind="stable")
    score_increase = sorted_pool.groupby("case_id", sort=False)["candidate_score"].diff().gt(1e-12)
    if score_increase.any():
        raise RuntimeError(f"{expected_method_label} scores increase with rank.")

    expected_target = frame["candidate_parent_asin"].astype(str).eq(
        frame["target_parent_asin"].astype(str)
    )
    if not boolean_series(frame["is_target"]).eq(expected_target).all():
        raise RuntimeError(
            f"{expected_method_label} is_target does not match candidate and target item IDs."
        )


def lists_from_long_pool(frame):
    ordered = frame.sort_values(["case_id", "candidate_rank"], kind="stable")
    return (
        ordered.groupby("case_id", sort=False)
        .agg(
            candidate_parent_asin_list=("candidate_parent_asin", list),
            candidate_brand_facet_text_list=("candidate_brand_facet_text", list),
            candidate_score_list=("candidate_score", list),
            candidate_count=("candidate_parent_asin", "size"),
            candidate_score_source=("candidate_score_source", "first"),
        )
        .reset_index()
    )


def long_from_list_pool(list_frame):
    work = list_frame.copy()
    work["candidate_rank_list"] = work["candidate_parent_asin_list"].map(
        lambda values: list(range(1, len(values) + 1))
    )
    long_frame = work.explode(
        [
            "candidate_parent_asin_list",
            "candidate_brand_facet_text_list",
            "candidate_score_list",
            "candidate_rank_list",
        ],
        ignore_index=True,
    ).rename(columns={
        "candidate_parent_asin_list": "candidate_parent_asin",
        "candidate_brand_facet_text_list": "candidate_brand_facet_text",
        "candidate_score_list": "candidate_score",
        "candidate_rank_list": "candidate_rank",
    })
    long_frame["candidate_parent_asin"] = long_frame["candidate_parent_asin"].astype(str)
    long_frame["candidate_brand_facet_text"] = (
        long_frame["candidate_brand_facet_text"].fillna("").astype(str)
    )
    long_frame["candidate_item_id"] = long_frame["candidate_parent_asin"]
    long_frame["candidate_rank"] = pd.to_numeric(long_frame["candidate_rank"], errors="raise").astype(int)
    long_frame["candidate_score"] = pd.to_numeric(long_frame["candidate_score"], errors="raise").astype(float)
    long_frame["is_target"] = long_frame["candidate_parent_asin"].eq(
        long_frame["target_parent_asin"].astype(str)
    )
    long_frame["is_gt"] = long_frame["is_target"].astype(np.int8)
    return long_frame


In [7]:
# ==== Resolve Both Winner Contracts and Validate Their Lineage ====
required_contract_paths = [
    QUERY_CACHE_PATH,
    ITEM_DOCS_PATH,
    STAGE1_WINNER_MANIFEST_PATH,
    PERSONALIZED_RUN_MANIFEST_PATH,
    PERSONALIZED_WINNER_MANIFEST_PATH,
]
missing_contract_paths = [str(path) for path in required_contract_paths if not path.exists()]
if missing_contract_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_contract_paths}")

stage1_winner = load_json(STAGE1_WINNER_MANIFEST_PATH)
personalized_run_manifest = load_json(PERSONALIZED_RUN_MANIFEST_PATH)
personalized_winner = load_json(PERSONALIZED_WINNER_MANIFEST_PATH)

if stage1_winner.get("contract_version") != QUERY_ONLY_WINNER_CONTRACT_VERSION:
    raise RuntimeError("Notebook 07 winner contract version mismatch.")
if personalized_winner.get("contract_version") != PERSONALIZED_WINNER_CONTRACT_VERSION:
    raise RuntimeError("Notebook 08 winner contract version mismatch.")
if stage1_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 07 winner contract category mismatch.")
if personalized_winner.get("category_id") != CATEGORY_ID:
    raise RuntimeError("Notebook 08 winner contract category mismatch.")
if stage1_winner.get("retrieval_evidence_scope") != EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 07 winner evidence scope mismatch.")
if stage1_winner.get("brand_graph_enabled") is not True:
    raise RuntimeError("Notebook 07 winner contract must preserve brand graph edges.")
if stage1_winner.get("brand_in_retrieval_text") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain brand in item text.")
if stage1_winner.get("brand_in_candidate_output") is not True:
    raise RuntimeError("Notebook 07 winner contract must retain candidate brand.")
if stage1_winner.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 07 winner contract must keep query-side brand matching disabled.")
if personalized_winner.get("query_only_retrieval_evidence_scope") != EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 08 winner baseline evidence scope mismatch.")
if personalized_run_manifest.get("evidence_scope") != EVIDENCE_SCOPE:
    raise RuntimeError("Notebook 08 run manifest evidence scope mismatch.")
if personalized_run_manifest.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Notebook 08 must use the Global Review item representation.")
if personalized_run_manifest.get("historical_review_reputation_used_as_profile_evidence") is not False:
    raise RuntimeError("Notebook 08 must keep historical review signals out of the QCHS profile source.")
if personalized_run_manifest.get("brand_profile_enabled") is not True:
    raise RuntimeError("Notebook 08 must enable brand in the user-profile source.")
if personalized_run_manifest.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 08 must keep query-side brand matching disabled.")
if personalized_run_manifest.get("fallback_policy") != "copy_baseline_candidates":
    raise RuntimeError("Notebook 08 run manifest must declare copy_baseline_candidates fallback policy.")
if personalized_run_manifest.get("fallback_changes_regime") is not False:
    raise RuntimeError("Notebook 08 fallback must not change regimes.")
if personalized_run_manifest.get("fallback_changes_case_universe") is not False:
    raise RuntimeError("Notebook 08 fallback must preserve the full case universe.")
if personalized_run_manifest.get("fallback_uses_all_prior") is not False:
    raise RuntimeError("Notebook 08 fallback must not use All Prior.")
if personalized_run_manifest.get("fallback_is_active_personalization") is not False:
    raise RuntimeError("Notebook 08 fallback must not be active personalization.")
if personalized_winner.get("historical_review_reputation_enabled") is not True:
    raise RuntimeError("Notebook 08 winner contract must enable Global Review item evidence.")
if personalized_winner.get("historical_review_reputation_used_as_profile_evidence") is not False:
    raise RuntimeError("Notebook 08 winner contract must exclude historical review signals from profile evidence.")
if personalized_winner.get("brand_profile_enabled") is not True:
    raise RuntimeError("Notebook 08 winner contract must enable brand in profile evidence.")
if personalized_winner.get("brand_query_matching_enabled") is not False:
    raise RuntimeError("Notebook 08 winner contract must keep query-side brand matching disabled.")

BASELINE_METHOD_SLUG = normalize_text(stage1_winner.get("winner_method_key"))
BASELINE_METHOD_LABEL = normalize_text(stage1_winner.get("winner_method_label"))
BASELINE_CANDIDATES_PATH = Path(normalize_text(stage1_winner.get("winner_candidate_path")))
PERSONALIZED_METHOD_SLUG = normalize_text(personalized_winner.get("winner_method_slug"))
PERSONALIZED_METHOD_LABEL = normalize_text(personalized_winner.get("winner_method_label"))
PERSONALIZED_CANDIDATE_LISTS_PATH = Path(
    normalize_text(personalized_winner.get("candidate_lists_path"))
)

if not all([
    BASELINE_METHOD_SLUG,
    BASELINE_METHOD_LABEL,
    PERSONALIZED_METHOD_SLUG,
    PERSONALIZED_METHOD_LABEL,
]):
    raise RuntimeError("A winner contract contains an empty method key or label.")
if PERSONALIZED_METHOD_SLUG == BASELINE_METHOD_SLUG:
    raise RuntimeError("The personalized winner must differ from the query-only winner method key.")
if personalized_winner.get("query_only_winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 08 winner contract does not reference the Notebook 07 winner.")
if personalized_winner.get("query_only_winner_method_label") != BASELINE_METHOD_LABEL:
    raise RuntimeError("Notebook 08 winner contract baseline label mismatch.")
if Path(personalized_winner.get("query_only_winner_candidate_path")) != BASELINE_CANDIDATES_PATH:
    raise RuntimeError("Notebook 08 winner contract baseline candidate path mismatch.")
if personalized_run_manifest.get("stage1_winner_method_key") != BASELINE_METHOD_SLUG:
    raise RuntimeError("Notebook 08 run manifest baseline method mismatch.")
if personalized_run_manifest.get("selected_method_slug") != PERSONALIZED_METHOD_SLUG:
    raise RuntimeError("Notebook 08 run manifest personalized winner mismatch.")
if personalized_run_manifest.get("selected_method") != PERSONALIZED_METHOD_LABEL:
    raise RuntimeError("Notebook 08 run manifest personalized winner label mismatch.")
if personalized_run_manifest.get("winner_contract_version") != PERSONALIZED_WINNER_CONTRACT_VERSION:
    raise RuntimeError("Notebook 08 run manifest winner-contract version mismatch.")
if personalized_run_manifest.get("stage1_winner_contract_version") != QUERY_ONLY_WINNER_CONTRACT_VERSION:
    raise RuntimeError("Notebook 08 run manifest query-only winner-contract version mismatch.")
if Path(personalized_run_manifest.get("output_paths", {}).get("candidate_lists", "")) != PERSONALIZED_CANDIDATE_LISTS_PATH:
    raise RuntimeError("Notebook 08 run manifest candidate-list path mismatch.")
expected_candidate_filter = {"column": "method_slug", "value": PERSONALIZED_METHOD_SLUG}
if personalized_winner.get("candidate_list_filter") != expected_candidate_filter:
    raise RuntimeError("Notebook 08 winner contract candidate-list filter mismatch.")

if stage1_winner.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 07 winner contract must use exact-K candidates.")
if personalized_winner.get("candidate_budget_policy") != "exact_k_all_methods":
    raise RuntimeError("Notebook 08 winner contract must use exact-K candidates.")
if stage1_winner.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 07 winner contract did not pass exact-K validation.")
if personalized_winner.get("exact_k_validation_passed") is not True:
    raise RuntimeError("Notebook 08 winner contract did not pass exact-K validation.")
if stage1_winner.get("candidate_budget_k") != MAX_CANDIDATE_DEPTH:
    raise RuntimeError("Notebook 07 winner contract candidate budget mismatch.")
if personalized_winner.get("candidate_budget_k") != MAX_CANDIDATE_DEPTH:
    raise RuntimeError("Notebook 08 winner contract candidate budget mismatch.")

source_paths = [BASELINE_CANDIDATES_PATH, PERSONALIZED_CANDIDATE_LISTS_PATH]
missing_source_paths = [str(path) for path in source_paths if not path.exists()]
if missing_source_paths:
    raise FileNotFoundError(f"Winner source artifacts are missing: {missing_source_paths}")

queries = pd.read_parquet(QUERY_CACHE_PATH).copy()
require_columns(
    queries,
    [
        "case_id", "user_id", "target_parent_asin", "regime",
        "target_selection_mode", "target_timestamp_ms", ACTIVE_QUERY_COLUMN,
    ],
    "query cache",
)
if queries["case_id"].duplicated().any():
    raise RuntimeError("The query cache contains duplicate case_id values.")
if "query_C" in queries.columns and not queries["query_C"].fillna("").astype(str).eq(
    queries[ACTIVE_QUERY_COLUMN].fillna("").astype(str)
).all():
    raise RuntimeError("query_C must be an exact compatibility alias of query when present.")

for column in ["case_id", "user_id", "target_parent_asin", "regime", "target_selection_mode"]:
    queries[column] = queries[column].fillna("").astype(str)
queries["target_timestamp_ms"] = pd.to_numeric(
    queries["target_timestamp_ms"], errors="raise"
).astype("int64")
queries["query"] = queries[ACTIVE_QUERY_COLUMN].map(normalize_text)
if queries["query"].eq("").any():
    raise RuntimeError("The active query column contains empty values.")

if "sampling_bracket" not in queries.columns:
    queries["sampling_bracket"] = queries["regime"]
queries["sampling_bracket"] = queries["sampling_bracket"].fillna("").astype(str)
optional_query_columns = [column for column in ["target_rank_desc"] if column in queries.columns]

query_metadata = queries[[
    "case_id", "user_id", "regime", "sampling_bracket", "target_selection_mode",
    "target_parent_asin", "target_timestamp_ms", "query", *optional_query_columns,
]].copy()
query_metadata["query_id"] = query_metadata["case_id"] + f"__{LEGACY_QUERY_METHOD_LABEL}"
query_metadata["query_text"] = query_metadata["query"]
query_metadata["query_method"] = LEGACY_QUERY_METHOD_LABEL
query_metadata["active_query_method"] = LEGACY_QUERY_METHOD_LABEL
query_metadata["active_query_column"] = ACTIVE_QUERY_COLUMN
query_metadata["gt_item_id"] = query_metadata["target_parent_asin"]
query_case_ids = set(query_metadata["case_id"].astype(str))

item_docs = pd.read_parquet(ITEM_DOCS_PATH, columns=["parent_asin", BRAND_TEXT_COLUMN]).copy()
item_docs["parent_asin"] = item_docs["parent_asin"].astype(str)
item_docs[BRAND_TEXT_COLUMN] = item_docs[BRAND_TEXT_COLUMN].fillna("").astype(str).map(normalize_text)
if item_docs["parent_asin"].duplicated().any():
    raise RuntimeError("Notebook 04 item documents contain duplicate parent_asin values.")
item_id_set = set(item_docs["parent_asin"])
item_brand_map = item_docs.set_index("parent_asin")[BRAND_TEXT_COLUMN]
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("Notebook 04 item documents must retain non-empty brand facets.")
expected_k = min(MAX_CANDIDATE_DEPTH, len(item_id_set))
if expected_k <= 0:
    raise RuntimeError("The item catalog is empty.")
if stage1_winner.get("effective_candidate_count_per_query") != expected_k:
    raise RuntimeError("Notebook 07 winner contract effective candidate count mismatch.")
if personalized_winner.get("effective_candidate_count_per_query") != expected_k:
    raise RuntimeError("Notebook 08 winner contract effective candidate count mismatch.")
if stage1_winner.get("catalog_size") != len(item_id_set):
    raise RuntimeError("Notebook 07 winner contract catalog size mismatch.")
if personalized_winner.get("catalog_size") != len(item_id_set):
    raise RuntimeError("Notebook 08 winner contract catalog size mismatch.")
if stage1_winner.get("query_count") != len(query_metadata):
    raise RuntimeError("Notebook 07 winner contract query count mismatch.")
if personalized_winner.get("query_count") != len(query_metadata):
    raise RuntimeError("Notebook 08 winner contract query count mismatch.")

baseline_raw = pd.read_parquet(BASELINE_CANDIDATES_PATH).copy()
require_columns(
    baseline_raw,
    [
        "case_id", "user_id", "regime", "target_parent_asin",
        "candidate_parent_asin", "candidate_rank", "candidate_score", "is_target",
        "method_key", "retrieval_method", "candidate_brand_facet_text",
    ],
    "Notebook 07 query-only winner candidates",
)
baseline_raw = baseline_raw.loc[
    baseline_raw["method_key"].fillna("").astype(str).eq(BASELINE_METHOD_SLUG)
].copy()
if baseline_raw.empty:
    raise RuntimeError("Notebook 07 winner candidate cache has no selected-method rows.")
observed_baseline_methods = set(baseline_raw["retrieval_method"].dropna().astype(str))
if observed_baseline_methods != {BASELINE_METHOD_LABEL}:
    raise RuntimeError(
        f"Expected only {BASELINE_METHOD_LABEL}, found {sorted(observed_baseline_methods)}."
    )

for column in ["case_id", "user_id", "regime", "target_parent_asin", "candidate_parent_asin"]:
    baseline_raw[column] = baseline_raw[column].fillna("").astype(str)
baseline_raw["candidate_rank"] = pd.to_numeric(
    baseline_raw["candidate_rank"], errors="raise"
).astype(int)
baseline_raw["candidate_score"] = pd.to_numeric(
    baseline_raw["candidate_score"], errors="raise"
).astype(float)
baseline_raw["is_target"] = boolean_series(baseline_raw["is_target"])
baseline_raw["candidate_brand_facet_text"] = baseline_raw["candidate_brand_facet_text"].fillna("").astype(str).map(normalize_text)
expected_baseline_brand = baseline_raw["candidate_parent_asin"].map(item_brand_map).fillna("").astype(str)
if not baseline_raw["candidate_brand_facet_text"].eq(expected_baseline_brand).all():
    raise RuntimeError("Notebook 07 candidate brand values do not match Notebook 04 item docs.")

personalized_all = pd.read_parquet(PERSONALIZED_CANDIDATE_LISTS_PATH).copy()
require_columns(
    personalized_all,
    [
        "case_id", "user_id", "regime", "target_parent_asin", "target_timestamp_ms",
        "prior_history_n", "query", "baseline_method_slug", "baseline_method_label",
        "method_slug", "method_label", "profile_fallback_flag", "profile_fallback_reason",
        "qchs_selected_prior_item_count", "qchs_profile_safe_facet_count",
        "qchs_profile_available", "raw_prior_item_count", "training_safe_prior_item_count",
        "qchs_brand_facet_count", "qchs_functional_facet_count",
        "candidate_parent_asin_list", "candidate_brand_facet_text_list", "candidate_score_list", "candidate_count",
        "candidate_score_source", "selected_personalized_method",
        "is_selected_personalized_method",
    ],
    "Notebook 08 candidate lists",
)
for column in [
    "case_id", "user_id", "regime", "target_parent_asin", "baseline_method_slug",
    "baseline_method_label", "method_slug", "method_label", "selected_personalized_method",
    "profile_fallback_reason",
]:
    personalized_all[column] = personalized_all[column].fillna("").astype(str)
personalized_all["target_timestamp_ms"] = pd.to_numeric(
    personalized_all["target_timestamp_ms"], errors="raise"
).astype("int64")
personalized_all["prior_history_n"] = pd.to_numeric(
    personalized_all["prior_history_n"], errors="raise"
).astype(int)
personalized_all["query"] = personalized_all["query"].map(normalize_text)
personalized_all["candidate_count"] = pd.to_numeric(
    personalized_all["candidate_count"], errors="raise"
).astype(int)
for column in [
    "qchs_selected_prior_item_count",
    "qchs_profile_safe_facet_count",
    "raw_prior_item_count",
    "training_safe_prior_item_count",
    "qchs_brand_facet_count",
    "qchs_functional_facet_count",
]:
    personalized_all[column] = pd.to_numeric(personalized_all[column], errors="raise").astype(int)
personalized_all["qchs_profile_available"] = boolean_series(personalized_all["qchs_profile_available"])
personalized_all["profile_fallback_flag"] = boolean_series(personalized_all["profile_fallback_flag"])

if not personalized_all["baseline_method_slug"].eq(BASELINE_METHOD_SLUG).all():
    raise RuntimeError("Notebook 08 candidate lists baseline method key mismatch.")
if not personalized_all["baseline_method_label"].eq(BASELINE_METHOD_LABEL).all():
    raise RuntimeError("Notebook 08 candidate lists baseline method label mismatch.")
if not personalized_all["selected_personalized_method"].eq(PERSONALIZED_METHOD_SLUG).all():
    raise RuntimeError("Notebook 08 candidate lists selected-personalized-method mismatch.")

baseline_rows_08 = personalized_all.loc[
    personalized_all["method_slug"].eq(BASELINE_METHOD_SLUG)
].copy()
personalized_rows = personalized_all.loc[
    personalized_all["method_slug"].eq(PERSONALIZED_METHOD_SLUG)
].copy()
selected_flag_rows = personalized_all.loc[boolean_series(personalized_all["is_selected_personalized_method"])]
if set(selected_flag_rows["method_slug"]) != {PERSONALIZED_METHOD_SLUG}:
    raise RuntimeError("Notebook 08 selected-personalized flags do not identify the winner method.")
if len(selected_flag_rows) != len(query_case_ids):
    raise RuntimeError("Notebook 08 must flag exactly one personalized-winner row per query.")
if set(selected_flag_rows["case_id"].astype(str)) != query_case_ids:
    raise RuntimeError("Notebook 08 personalized-winner flags do not cover the full query set.")

for frame_name, frame in [
    ("Notebook 08 query-only winner", baseline_rows_08),
    ("Notebook 08 personalized winner", personalized_rows),
]:
    if frame.duplicated("case_id").any():
        raise RuntimeError(f"{frame_name} contains duplicate case rows.")
    if set(frame["case_id"].astype(str)) != query_case_ids:
        raise RuntimeError(f"{frame_name} case IDs do not match the active query cache.")
    for row in frame.itertuples(index=False):
        validate_ranked_list(
            row.candidate_parent_asin_list,
            row.candidate_score_list,
            item_id_set,
            expected_k,
            f"{frame_name} case_id={row.case_id}",
        )
        candidate_items = [str(value) for value in as_list(row.candidate_parent_asin_list)]
        candidate_brands = [normalize_text(value) for value in as_list(row.candidate_brand_facet_text_list)]
        if len(candidate_brands) != expected_k:
            raise RuntimeError(f"{frame_name} candidate brand count mismatch for case_id={row.case_id}.")
        expected_brands = [normalize_text(item_brand_map.get(item_id, "")) for item_id in candidate_items]
        if candidate_brands != expected_brands:
            raise RuntimeError(f"{frame_name} candidate brand values mismatch for case_id={row.case_id}.")
        if int(row.candidate_count) != expected_k:
            raise RuntimeError(f"{frame_name} candidate_count mismatch for case_id={row.case_id}.")

prior_count_map = baseline_rows_08.set_index("case_id")["prior_history_n"]
query_metadata["prior_history_n"] = query_metadata["case_id"].map(prior_count_map)
if query_metadata["prior_history_n"].isna().any():
    raise RuntimeError("Notebook 08 did not provide prior_history_n for every query case.")
query_metadata["prior_history_n"] = query_metadata["prior_history_n"].astype(int)
personalized_prior_map = personalized_rows.set_index("case_id")["prior_history_n"]
if not query_metadata.set_index("case_id")["prior_history_n"].eq(personalized_prior_map).all():
    raise RuntimeError("Notebook 08 baseline and personalized rows have different prior_history_n values.")

expected_meta = query_metadata.set_index("case_id")
for column in ["user_id", "regime", "target_parent_asin"]:
    expected_values = baseline_raw["case_id"].map(expected_meta[column])
    mismatch = baseline_raw[column].ne(expected_values)
    if mismatch.any():
        raise RuntimeError(
            f"Notebook 07 query-only winner metadata mismatch in {column}: {int(mismatch.sum())} rows."
        )

for frame_name, frame in [
    ("Notebook 08 query-only winner", baseline_rows_08),
    ("Notebook 08 personalized winner", personalized_rows),
]:
    for column in ["user_id", "regime", "target_parent_asin", "target_timestamp_ms", "query"]:
        expected_values = frame["case_id"].map(expected_meta[column])
        observed = frame[column]
        if column == "query":
            observed = observed.map(normalize_text)
            expected_values = expected_values.map(normalize_text)
        mismatch = observed.ne(expected_values)
        if mismatch.any():
            raise RuntimeError(
                f"{frame_name} metadata mismatch in {column}: {int(mismatch.sum())} rows."
            )

baseline_meta_columns = [
    "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "target_parent_asin", "gt_item_id",
    "target_timestamp_ms", "prior_history_n", "query", "query_text",
    "query_method", "active_query_method", "active_query_column",
    *optional_query_columns,
]
baseline_long = baseline_raw[[
    "case_id", "candidate_parent_asin", "candidate_brand_facet_text", "candidate_rank", "candidate_score", "is_target",
    *(["score_semantics"] if "score_semantics" in baseline_raw.columns else []),
]].merge(
    query_metadata[baseline_meta_columns],
    on="case_id",
    how="left",
    validate="many_to_one",
)
baseline_long["category_id"] = CATEGORY_ID
baseline_long["method_slug"] = BASELINE_METHOD_SLUG
baseline_long["method"] = BASELINE_METHOD_LABEL
baseline_long["method_label"] = BASELINE_METHOD_LABEL
baseline_long["retrieval_method"] = BASELINE_METHOD_LABEL
baseline_long["retrieval_method_label"] = BASELINE_METHOD_LABEL
baseline_long["baseline_retrieval_method_key"] = BASELINE_METHOD_SLUG
baseline_long["baseline_retrieval_method"] = BASELINE_METHOD_LABEL
baseline_long["baseline_retrieval_method_label"] = BASELINE_METHOD_LABEL
baseline_long["candidate_pool_type"] = BASELINE_METHOD_LABEL
baseline_long["candidate_pool_role"] = "baseline_query_only"
baseline_long["profile_facet_policy"] = ""
baseline_long["profile_fallback_flag"] = 0
baseline_long["candidate_score_source"] = (
    baseline_long["score_semantics"].fillna(f"notebook07_{BASELINE_METHOD_SLUG}_score").astype(str)
    if "score_semantics" in baseline_long.columns
    else f"notebook07_{BASELINE_METHOD_SLUG}_score"
)
baseline_long["selected_personalized_method"] = PERSONALIZED_METHOD_SLUG
baseline_long["selected_personalized_method_label"] = PERSONALIZED_METHOD_LABEL
baseline_long["is_selected_personalized_method"] = 0
baseline_long["candidate_item_id"] = baseline_long["candidate_parent_asin"]
baseline_long["is_gt"] = baseline_long["is_target"].astype(np.int8)

baseline_lists_for_check = lists_from_long_pool(baseline_long)[[
    "case_id", "candidate_parent_asin_list", "candidate_score_list"
]].set_index("case_id")
baseline_rows_08_check = baseline_rows_08.set_index("case_id")
for case_id in sorted(query_case_ids):
    baseline_items = baseline_lists_for_check.at[case_id, "candidate_parent_asin_list"]
    baseline_scores = np.asarray(
        baseline_lists_for_check.at[case_id, "candidate_score_list"], dtype=float
    )
    notebook08_items = [
        str(value)
        for value in as_list(baseline_rows_08_check.at[case_id, "candidate_parent_asin_list"])
    ]
    notebook08_scores = np.asarray(
        as_list(baseline_rows_08_check.at[case_id, "candidate_score_list"]), dtype=float
    )
    if baseline_items != notebook08_items:
        raise RuntimeError(
            f"Notebook 08 query-only winner order differs from Notebook 07 for case_id={case_id}."
        )
    if not np.allclose(baseline_scores, notebook08_scores, rtol=1e-6, atol=1e-8):
        raise RuntimeError(
            f"Notebook 08 query-only winner scores differ from Notebook 07 for case_id={case_id}."
        )

print("Rows: queries", len(query_metadata))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)
print("Rows: Notebook 07 winner candidates", len(baseline_long))
print("Rows: Notebook 08 personalized winner lists", len(personalized_rows))
print("Validation status: PASS")


Rows: queries 1968
Query-only winner: hybrid_dense_bm25 - Dense-BM25 Hybrid
Personalized winner: profile_sparse_qchs - Profile Sparse QCHS
Rows: Notebook 07 winner candidates 1968000
Rows: Notebook 08 personalized winner lists 1968
Validation status: PASS


In [8]:
# ==== Materialize the Two Winner-Locked Pool Formats ====
baseline_list = query_metadata[baseline_meta_columns].merge(
    lists_from_long_pool(baseline_long),
    on="case_id",
    how="left",
    validate="one_to_one",
)
baseline_list["category_id"] = CATEGORY_ID
baseline_list["method_slug"] = BASELINE_METHOD_SLUG
baseline_list["method"] = BASELINE_METHOD_LABEL
baseline_list["method_label"] = BASELINE_METHOD_LABEL
baseline_list["retrieval_method"] = BASELINE_METHOD_LABEL
baseline_list["retrieval_method_label"] = BASELINE_METHOD_LABEL
baseline_list["baseline_retrieval_method_key"] = BASELINE_METHOD_SLUG
baseline_list["baseline_retrieval_method"] = BASELINE_METHOD_LABEL
baseline_list["baseline_retrieval_method_label"] = BASELINE_METHOD_LABEL
baseline_list["candidate_pool_type"] = BASELINE_METHOD_LABEL
baseline_list["candidate_pool_role"] = "baseline_query_only"
baseline_list["profile_facet_policy"] = ""
baseline_list["profile_fallback_flag"] = 0
baseline_list["selected_personalized_method"] = PERSONALIZED_METHOD_SLUG
baseline_list["selected_personalized_method_label"] = PERSONALIZED_METHOD_LABEL
baseline_list["is_selected_personalized_method"] = 0

profile_optional_columns = [
    column for column in [
        "raw_prior_item_count",
        "training_safe_prior_item_count",
        "qchs_selected_prior_item_count",
        "qchs_profile_safe_facet_count",
        "qchs_profile_available",
        "profile_fallback_reason",
        "qchs_brand_facet_count",
        "qchs_functional_facet_count",
        "profile_selected_prior_item_count",
        "profile_selected_facet_count",
        "profile_anchor_phrase_count",
        "dense_qchs_candidate_count",
        "sparse_qchs_candidate_count",
        "graph_qchs_candidate_count",
    ]
    if column in personalized_rows.columns
]
profile_list = personalized_rows[[
    "case_id", "candidate_parent_asin_list", "candidate_brand_facet_text_list", "candidate_score_list", "candidate_count",
    "candidate_score_source", "profile_fallback_flag", *profile_optional_columns,
]].merge(
    query_metadata[baseline_meta_columns],
    on="case_id",
    how="left",
    validate="one_to_one",
)
profile_list["category_id"] = CATEGORY_ID
profile_list["method_slug"] = PERSONALIZED_METHOD_SLUG
profile_list["method"] = PERSONALIZED_METHOD_LABEL
profile_list["method_label"] = PERSONALIZED_METHOD_LABEL
profile_list["retrieval_method"] = PERSONALIZED_METHOD_LABEL
profile_list["retrieval_method_label"] = PERSONALIZED_METHOD_LABEL
profile_list["baseline_retrieval_method_key"] = BASELINE_METHOD_SLUG
profile_list["baseline_retrieval_method"] = BASELINE_METHOD_LABEL
profile_list["baseline_retrieval_method_label"] = BASELINE_METHOD_LABEL
profile_list["candidate_pool_type"] = PERSONALIZED_METHOD_LABEL
profile_list["candidate_pool_role"] = "personalized_retrieval"
profile_list["profile_facet_policy"] = "qchs_selected_prior_items_profile_safe_functional_and_brand_facets"
profile_list["selected_personalized_method"] = PERSONALIZED_METHOD_SLUG
profile_list["selected_personalized_method_label"] = PERSONALIZED_METHOD_LABEL
profile_list["is_selected_personalized_method"] = 1

list_column_order = [
    "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "target_parent_asin", "gt_item_id", "target_timestamp_ms",
    "prior_history_n", "query", "query_text", "query_method", "active_query_method",
    "active_query_column", *optional_query_columns,
    "baseline_retrieval_method_key", "baseline_retrieval_method",
    "baseline_retrieval_method_label", "method", "method_slug", "method_label",
    "retrieval_method", "retrieval_method_label", "candidate_pool_type",
    "candidate_pool_role", "profile_facet_policy", "profile_fallback_flag",
    *profile_optional_columns,
    "candidate_parent_asin_list", "candidate_brand_facet_text_list", "candidate_score_list", "candidate_count",
    "candidate_score_source", "selected_personalized_method",
    "selected_personalized_method_label", "is_selected_personalized_method",
]
list_column_order = list(dict.fromkeys(list_column_order))
baseline_list = baseline_list[[
    column for column in list_column_order if column in baseline_list.columns
]].copy()
profile_list = profile_list[[
    column for column in list_column_order if column in profile_list.columns
]].copy()

baseline_long = long_from_list_pool(baseline_list)
profile_long = long_from_list_pool(profile_list)

long_column_order = [
    column for column in list_column_order
    if column not in {"candidate_parent_asin_list", "candidate_brand_facet_text_list", "candidate_score_list", "candidate_count"}
] + [
    "candidate_parent_asin", "candidate_brand_facet_text", "candidate_item_id", "candidate_rank", "candidate_score",
    "is_target", "is_gt",
]
baseline_long = baseline_long[[
    column for column in long_column_order if column in baseline_long.columns
]].copy()
profile_long = profile_long[[
    column for column in long_column_order if column in profile_long.columns
]].copy()

validate_long_pool(
    baseline_long,
    BASELINE_METHOD_SLUG,
    BASELINE_METHOD_LABEL,
    query_case_ids,
    item_id_set,
    expected_k,
)
validate_long_pool(
    profile_long,
    PERSONALIZED_METHOD_SLUG,
    PERSONALIZED_METHOD_LABEL,
    query_case_ids,
    item_id_set,
    expected_k,
)

for frame_name, frame in [
    ("query-only winner", baseline_long),
    ("personalized winner", profile_long),
]:
    expected_brand = frame["candidate_parent_asin"].astype(str).map(item_brand_map).fillna("").astype(str)
    if not frame["candidate_brand_facet_text"].fillna("").astype(str).eq(expected_brand).all():
        raise RuntimeError(f"{frame_name} candidate brand values do not match Notebook 04 item docs.")

print("Rows: query-only winner", len(baseline_long))
print("Rows: personalized winner", len(profile_long))
print("Validation status: PASS")


Rows: query-only winner 1968000
Rows: personalized winner 1968000
Validation status: PASS


In [9]:
# ==== Export Pools and Stage-1 Descriptive Summaries ====
baseline_list.to_parquet(QUERY_ONLY_WINNER_LIST_PATH, index=False)
baseline_long.to_parquet(QUERY_ONLY_WINNER_POOL_PATH, index=False)
profile_list.to_parquet(PERSONALIZED_WINNER_LIST_PATH, index=False)
profile_long.to_parquet(PERSONALIZED_WINNER_POOL_PATH, index=False)

pool_summary = pd.DataFrame([
    {
        "category_id": CATEGORY_ID,
        "method_slug": BASELINE_METHOD_SLUG,
        "retrieval_method": BASELINE_METHOD_LABEL,
        "candidate_pool_role": "baseline_query_only",
        "case_count": int(baseline_list["case_id"].nunique()),
        "candidate_row_count": int(len(baseline_long)),
        "candidate_count_min": int(baseline_list["candidate_count"].min()),
        "candidate_count_mean": float(baseline_list["candidate_count"].mean()),
        "candidate_count_max": int(baseline_list["candidate_count"].max()),
        "target_hit_rate_at_1000": float(
            baseline_long.groupby("case_id")["is_target"].max().mean()
        ),
        "long_pool_path": str(QUERY_ONLY_WINNER_POOL_PATH),
        "list_pool_path": str(QUERY_ONLY_WINNER_LIST_PATH),
    },
    {
        "category_id": CATEGORY_ID,
        "method_slug": PERSONALIZED_METHOD_SLUG,
        "retrieval_method": PERSONALIZED_METHOD_LABEL,
        "candidate_pool_role": "personalized_retrieval",
        "case_count": int(profile_list["case_id"].nunique()),
        "candidate_row_count": int(len(profile_long)),
        "candidate_count_min": int(profile_list["candidate_count"].min()),
        "candidate_count_mean": float(profile_list["candidate_count"].mean()),
        "candidate_count_max": int(profile_list["candidate_count"].max()),
        "target_hit_rate_at_1000": float(
            profile_long.groupby("case_id")["is_target"].max().mean()
        ),
        "long_pool_path": str(PERSONALIZED_WINNER_POOL_PATH),
        "list_pool_path": str(PERSONALIZED_WINNER_LIST_PATH),
    },
])
pool_summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")


def target_rank_by_case(pool_df):
    target_rows = pool_df.loc[pool_df["is_target"].astype(bool), ["case_id", "candidate_rank"]].copy()
    if target_rows.duplicated("case_id").any():
        raise RuntimeError("A candidate pool contains duplicate target rows for a case.")
    return target_rows.set_index("case_id")["candidate_rank"].astype(float)


def rank_metric_values(target_ranks, depth):
    ranks = pd.to_numeric(target_ranks, errors="coerce")
    hit = ranks.notna() & ranks.le(depth)
    rr = pd.Series(0.0, index=ranks.index)
    rr.loc[hit] = 1.0 / ranks.loc[hit]
    ndcg = pd.Series(0.0, index=ranks.index)
    ndcg.loc[hit] = 1.0 / np.log2(ranks.loc[hit] + 1.0)
    return hit.astype(float), rr.astype(float), ndcg.astype(float)


baseline_target_ranks = target_rank_by_case(baseline_long)
profile_target_ranks = target_rank_by_case(profile_long)
all_case_index = pd.Index(sorted(query_case_ids), name="case_id")
baseline_target_ranks = baseline_target_ranks.reindex(all_case_index)
profile_target_ranks = profile_target_ranks.reindex(all_case_index)

stage1_uplift_by_case = query_metadata.set_index("case_id")[["regime"]].reindex(all_case_index).reset_index()
for depth in [1, 5, 1000]:
    baseline_hit, baseline_mrr, baseline_ndcg = rank_metric_values(baseline_target_ranks, depth)
    profile_hit, profile_mrr, profile_ndcg = rank_metric_values(profile_target_ranks, depth)
    stage1_uplift_by_case[f"baseline_HitRate@{depth}"] = baseline_hit.reindex(all_case_index).to_numpy()
    stage1_uplift_by_case[f"profile_HitRate@{depth}"] = profile_hit.reindex(all_case_index).to_numpy()
    stage1_uplift_by_case[f"baseline_MRR@{depth}"] = baseline_mrr.reindex(all_case_index).to_numpy()
    stage1_uplift_by_case[f"profile_MRR@{depth}"] = profile_mrr.reindex(all_case_index).to_numpy()
    stage1_uplift_by_case[f"baseline_NDCG@{depth}"] = baseline_ndcg.reindex(all_case_index).to_numpy()
    stage1_uplift_by_case[f"profile_NDCG@{depth}"] = profile_ndcg.reindex(all_case_index).to_numpy()

metric_pairs = [
    ("HitRate@1000", "baseline_HitRate@1000", "profile_HitRate@1000"),
    ("MRR@1000", "baseline_MRR@1000", "profile_MRR@1000"),
    ("NDCG@1000", "baseline_NDCG@1000", "profile_NDCG@1000"),
    ("NDCG@5", "baseline_NDCG@5", "profile_NDCG@5"),
    ("NDCG@1", "baseline_NDCG@1", "profile_NDCG@1"),
]

overall_uplift_rows = []
for metric_name, baseline_col, profile_col in metric_pairs:
    baseline_value = float(stage1_uplift_by_case[baseline_col].mean())
    profile_value = float(stage1_uplift_by_case[profile_col].mean())
    overall_uplift_rows.append({
        "category_id": CATEGORY_ID,
        "metric": metric_name,
        "baseline_retrieval_method_key": BASELINE_METHOD_SLUG,
        "personalized_retrieval_method_key": PERSONALIZED_METHOD_SLUG,
        "baseline_value": baseline_value,
        "personalized_value": profile_value,
        "personalized_minus_baseline": profile_value - baseline_value,
    })
stage1_personalization_uplift_overall = pd.DataFrame(overall_uplift_rows)

by_regime_rows = []
for regime, group in stage1_uplift_by_case.groupby("regime", sort=True):
    for metric_name, baseline_col, profile_col in metric_pairs:
        baseline_value = float(group[baseline_col].mean())
        profile_value = float(group[profile_col].mean())
        by_regime_rows.append({
            "category_id": CATEGORY_ID,
            "regime": regime,
            "metric": metric_name,
            "baseline_retrieval_method_key": BASELINE_METHOD_SLUG,
            "personalized_retrieval_method_key": PERSONALIZED_METHOD_SLUG,
            "baseline_value": baseline_value,
            "personalized_value": profile_value,
            "personalized_minus_baseline": profile_value - baseline_value,
        })
stage1_personalization_uplift_by_regime = pd.DataFrame(by_regime_rows)

cold_case_ids = set(
    query_metadata.loc[
        query_metadata["regime"].eq("cold")
        & pd.to_numeric(query_metadata["prior_history_n"], errors="raise").eq(0),
        "case_id",
    ].astype(str)
)
cold_label_with_history = query_metadata.loc[
    query_metadata["regime"].eq("cold")
    & pd.to_numeric(query_metadata["prior_history_n"], errors="raise").ne(0),
    "case_id",
].astype(str).tolist()
if cold_label_with_history:
    raise RuntimeError(
        "Cold regime rows have nonzero strict prior history in Notebook 09: "
        f"{cold_label_with_history[:10]}"
    )

baseline_case_count = int(baseline_list["case_id"].nunique())
personalized_case_count = int(profile_list["case_id"].nunique())
if set(baseline_list["case_id"].astype(str)) != query_case_ids:
    raise RuntimeError("Baseline candidate pool must preserve the complete query case universe.")
if set(profile_list["case_id"].astype(str)) != query_case_ids:
    raise RuntimeError("Personalized candidate pool must preserve the complete query case universe.")
if baseline_case_count != personalized_case_count:
    raise RuntimeError("Baseline and personalized candidate pools must contain the same case count.")

query_regime_counts = query_metadata["regime"].value_counts().sort_index().astype(int).to_dict()
baseline_regime_counts = baseline_list["regime"].value_counts().sort_index().astype(int).to_dict()
profile_regime_counts = profile_list["regime"].value_counts().sort_index().astype(int).to_dict()
regime_counts_unchanged = (
    baseline_regime_counts == query_regime_counts
    and profile_regime_counts == query_regime_counts
)
if not regime_counts_unchanged:
    raise RuntimeError(
        "Candidate-pool regime counts changed: "
        f"query={query_regime_counts}, baseline={baseline_regime_counts}, profile={profile_regime_counts}"
    )

fallback_case_ids = set(
    profile_list.loc[boolean_series(profile_list["profile_fallback_flag"]), "case_id"].astype(str)
)
fallback_case_count = int(len(fallback_case_ids))

identity_columns = ["case_id", "candidate_rank"]
baseline_fallback = baseline_long.loc[
    baseline_long["case_id"].astype(str).isin(fallback_case_ids),
    [*identity_columns, "candidate_parent_asin", "candidate_score", "is_target"],
].sort_values(identity_columns).reset_index(drop=True)
profile_fallback = profile_long.loc[
    profile_long["case_id"].astype(str).isin(fallback_case_ids),
    [*identity_columns, "candidate_parent_asin", "candidate_score", "is_target"],
].sort_values(identity_columns).reset_index(drop=True)

expected_fallback_rows = fallback_case_count * expected_k
item_order_mismatch_count = 0
rank_mismatch_count = 0
score_mismatch_count = 0
target_flag_mismatch_count = 0
if len(baseline_fallback) != expected_fallback_rows or len(profile_fallback) != expected_fallback_rows:
    fallback_candidate_identity_validated = False
else:
    item_order_mismatch_count = int(
        baseline_fallback["candidate_parent_asin"].astype(str).ne(
            profile_fallback["candidate_parent_asin"].astype(str)
        ).sum()
    )
    rank_mismatch_count = int(
        baseline_fallback["candidate_rank"].astype(int).ne(
            profile_fallback["candidate_rank"].astype(int)
        ).sum()
    )
    target_flag_mismatch_count = int(
        baseline_fallback["is_target"].astype(bool).ne(
            profile_fallback["is_target"].astype(bool)
        ).sum()
    )
    score_close = np.isclose(
        baseline_fallback["candidate_score"].astype(float).to_numpy(),
        profile_fallback["candidate_score"].astype(float).to_numpy(),
        rtol=1e-12,
        atol=1e-12,
    )
    score_mismatch_count = int((~score_close).sum())
    fallback_candidate_identity_validated = (
        item_order_mismatch_count == 0
        and rank_mismatch_count == 0
        and score_mismatch_count == 0
        and target_flag_mismatch_count == 0
    )

cold_candidate_fallback_qc = pd.DataFrame([{
    "category_id": CATEGORY_ID,
    "baseline_retrieval_method_key": BASELINE_METHOD_SLUG,
    "personalized_retrieval_method_key": PERSONALIZED_METHOD_SLUG,
    "n_cold_queries": int(len(cold_case_ids)),
    "n_fallback_queries": fallback_case_count,
    "expected_rows": int(expected_fallback_rows),
    "actual_rows": int(len(profile_fallback)),
    "item_order_mismatch_count": item_order_mismatch_count,
    "rank_mismatch_count": rank_mismatch_count,
    "score_mismatch_count": score_mismatch_count,
    "target_flag_mismatch_count": target_flag_mismatch_count,
    "exact_identity_passed": bool(fallback_candidate_identity_validated),
}])
if not fallback_candidate_identity_validated:
    raise RuntimeError("Notebook 09 fallback query-only and personalized winner candidates differ.")

fallback_uplift = stage1_uplift_by_case.loc[
    stage1_uplift_by_case["case_id"].astype(str).isin(fallback_case_ids)
]
for column in [
    "HitRate@1000",
    "MRR@1000",
    "NDCG@1000",
    "NDCG@5",
    "NDCG@1",
]:
    baseline_col = f"baseline_{column}"
    profile_col = f"profile_{column}"
    if not fallback_uplift.empty and (fallback_uplift[profile_col] - fallback_uplift[baseline_col]).abs().gt(1e-12).any():
        raise RuntimeError("Fallback Stage 1 personalization uplift must be exactly zero.")

cold_candidate_fallback_qc.to_csv(COLD_CANDIDATE_FALLBACK_QC_PATH, index=False, encoding="utf-8-sig")
stage1_personalization_uplift_overall.to_csv(
    STAGE1_PERSONALIZATION_UPLIFT_OVERALL_PATH, index=False, encoding="utf-8-sig"
)
stage1_personalization_uplift_by_regime.to_csv(
    STAGE1_PERSONALIZATION_UPLIFT_BY_REGIME_PATH, index=False, encoding="utf-8-sig"
)

print("Baseline cases:", baseline_case_count)
print("Personalized cases:", personalized_case_count)
print("Fallback cases preserved:", fallback_case_count)
print("Regime counts unchanged: passed")
print("Fallback candidate identity: passed")
print("Output:", QUERY_ONLY_WINNER_POOL_PATH)
print("Output:", PERSONALIZED_WINNER_POOL_PATH)
print("Rows: exported", int(len(baseline_long) + len(profile_long)))


Baseline cases: 1968
Personalized cases: 1968
Fallback cases preserved: 1415
Regime counts unchanged: passed
Fallback candidate identity: passed
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/query_only_winner_top1000_herbal.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/by_method/personalized_winner_top1000_herbal.parquet
Rows: exported 3936000


In [10]:
# ==== Reload, Validate, and Write the Stage-Transition Manifest ====
exported_paths = [
    QUERY_ONLY_WINNER_POOL_PATH,
    QUERY_ONLY_WINNER_LIST_PATH,
    PERSONALIZED_WINNER_POOL_PATH,
    PERSONALIZED_WINNER_LIST_PATH,
    SUMMARY_PATH,
    COLD_CANDIDATE_FALLBACK_QC_PATH,
    STAGE1_PERSONALIZATION_UPLIFT_OVERALL_PATH,
    STAGE1_PERSONALIZATION_UPLIFT_BY_REGIME_PATH,
]
missing_outputs = [str(path) for path in exported_paths if not path.exists()]
if missing_outputs:
    raise RuntimeError(f"Missing candidate-pool outputs: {missing_outputs}")

reloaded_baseline = pd.read_parquet(QUERY_ONLY_WINNER_POOL_PATH)
reloaded_profile = pd.read_parquet(PERSONALIZED_WINNER_POOL_PATH)
validate_long_pool(
    reloaded_baseline,
    BASELINE_METHOD_SLUG,
    BASELINE_METHOD_LABEL,
    query_case_ids,
    item_id_set,
    expected_k,
)
validate_long_pool(
    reloaded_profile,
    PERSONALIZED_METHOD_SLUG,
    PERSONALIZED_METHOD_LABEL,
    query_case_ids,
    item_id_set,
    expected_k,
)

required_compatibility_columns = {
    "category_id", "case_id", "query_id", "user_id", "regime", "sampling_bracket",
    "target_selection_mode", "query_method", "active_query_method", "gt_item_id",
    "target_parent_asin", "query_text", "prior_history_n",
    "baseline_retrieval_method_key", "baseline_retrieval_method", "method",
    "method_slug", "retrieval_method", "retrieval_method_label",
    "candidate_parent_asin", "candidate_rank", "candidate_score", "is_target", "is_gt",
}
for method_name, frame in [
    (BASELINE_METHOD_LABEL, reloaded_baseline),
    (PERSONALIZED_METHOD_LABEL, reloaded_profile),
]:
    missing_columns = sorted(required_compatibility_columns.difference(frame.columns))
    if missing_columns:
        raise RuntimeError(f"{method_name} export is missing downstream columns: {missing_columns}")

export_qc = pd.DataFrame([
    {"check": "active_query_column", "status": "PASS", "value": ACTIVE_QUERY_COLUMN},
    {"check": "query_case_set_matches", "status": "PASS", "value": len(query_case_ids)},
    {
        "check": "query_only_winner",
        "status": "PASS",
        "value": f"{BASELINE_METHOD_SLUG}:{BASELINE_METHOD_LABEL}",
    },
    {
        "check": "personalized_winner",
        "status": "PASS",
        "value": f"{PERSONALIZED_METHOD_SLUG}:{PERSONALIZED_METHOD_LABEL}",
    },
    {
        "check": "notebook08_baseline_matches_notebook07",
        "status": "PASS",
        "value": len(query_case_ids),
    },
    {"check": "candidate_budget_policy", "status": "PASS", "value": "exact_k_all_methods"},
    {"check": "candidate_depth", "status": "PASS", "value": expected_k},
    {"check": "variable_candidate_count_allowed", "status": "PASS", "value": False},
    {"check": "fallback_policy", "status": "PASS", "value": "baseline_equivalent_no_op"},
    {"check": "fallback_cases_preserved", "status": "PASS", "value": fallback_case_count},
    {"check": "fallback_candidate_identity", "status": "PASS", "value": bool(fallback_candidate_identity_validated)},
])
export_qc.to_csv(QC_PATH, index=False, encoding="utf-8-sig")

manifest = {
    "notebook_name": f"09_retrieval_candidate_pool_export_{CATEGORY_ID}.ipynb",
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "evidence_scope": EVIDENCE_SCOPE,
    "query_evidence_scope": QUERY_EVIDENCE_SCOPE,
    "historical_review_reputation_enabled": True,
    "historical_review_reputation_used_as_profile_evidence": False,
    "brand_graph_enabled": True,
    "brand_in_retrieval_text": True,
    "brand_in_candidate_output": True,
    "brand_profile_enabled": True,
    "brand_query_matching_enabled": False,
    "brand_in_synthetic_query": False,
    "active_query_column": ACTIVE_QUERY_COLUMN,
    "legacy_query_method_label": LEGACY_QUERY_METHOD_LABEL,
    "query_only_winner_contract_version": QUERY_ONLY_WINNER_CONTRACT_VERSION,
    "personalized_winner_contract_version": PERSONALIZED_WINNER_CONTRACT_VERSION,
    "baseline_retrieval_winner_method_key": BASELINE_METHOD_SLUG,
    "baseline_retrieval_winner_method_label": BASELINE_METHOD_LABEL,
    "selected_personalized_method_slug": PERSONALIZED_METHOD_SLUG,
    "selected_personalized_method_label": PERSONALIZED_METHOD_LABEL,
    "methods_exported": [BASELINE_METHOD_LABEL, PERSONALIZED_METHOD_LABEL],
    "candidate_budget_policy": "exact_k_all_methods",
    "candidate_budget_k": MAX_CANDIDATE_DEPTH,
    "effective_candidate_count_per_query": expected_k,
    "variable_candidate_count_allowed": False,
    "candidate_scores_preserved": True,
    "retrieval_recomputed": False,
    "complete_case_universe_preserved": True,
    "regime_counts_unchanged": True,
    "qchs_fallback_cases_preserved": True,
    "fallback_candidate_identity_validated": True,
    "fallback_cases_removed": 0,
    "fallback_policy": "baseline_equivalent_no_op",
    "cold_retrieval_fallback_policy": "exact_query_only_winner_copy",
    "cold_retrieval_identity_passed": bool(fallback_candidate_identity_validated),
    "n_cold_queries": int(len(cold_case_ids)),
    "fallback_case_count": fallback_case_count,
    "user_prior_enabled": {
        BASELINE_METHOD_SLUG: False,
        PERSONALIZED_METHOD_SLUG: True,
    },
    "input_paths": {
        "query_cache": str(QUERY_CACHE_PATH),
        "item_docs": str(ITEM_DOCS_PATH),
        "notebook07_winner_contract": str(STAGE1_WINNER_MANIFEST_PATH),
        "notebook07_winner_candidates": str(BASELINE_CANDIDATES_PATH),
        "notebook08_run_manifest": str(PERSONALIZED_RUN_MANIFEST_PATH),
        "notebook08_winner_contract": str(PERSONALIZED_WINNER_MANIFEST_PATH),
        "notebook08_candidate_lists": str(PERSONALIZED_CANDIDATE_LISTS_PATH),
    },
    "output_paths": {
        "query_only_winner_long": str(QUERY_ONLY_WINNER_POOL_PATH),
        "query_only_winner_list": str(QUERY_ONLY_WINNER_LIST_PATH),
        "personalized_winner_long": str(PERSONALIZED_WINNER_POOL_PATH),
        "personalized_winner_list": str(PERSONALIZED_WINNER_LIST_PATH),
        "summary": str(SUMMARY_PATH),
        "qc": str(QC_PATH),
        "cold_candidate_fallback_qc": str(COLD_CANDIDATE_FALLBACK_QC_PATH),
        "stage1_personalization_uplift_overall": str(STAGE1_PERSONALIZATION_UPLIFT_OVERALL_PATH),
        "stage1_personalization_uplift_by_regime": str(STAGE1_PERSONALIZATION_UPLIFT_BY_REGIME_PATH),
    },
    "query_count": int(len(query_case_ids)),
    "candidate_rows": {
        BASELINE_METHOD_SLUG: int(len(reloaded_baseline)),
        PERSONALIZED_METHOD_SLUG: int(len(reloaded_profile)),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

required_final_outputs = [*exported_paths, QC_PATH, MANIFEST_PATH]
missing_final_outputs = [str(path) for path in required_final_outputs if not path.exists()]
if missing_final_outputs:
    raise RuntimeError(f"Missing final Notebook 09 outputs: {missing_final_outputs}")

print("Output:", MANIFEST_PATH)
print("Rows: queries", len(query_case_ids))
print("Query-only winner:", BASELINE_METHOD_SLUG, "-", BASELINE_METHOD_LABEL)
print("Personalized winner:", PERSONALIZED_METHOD_SLUG, "-", PERSONALIZED_METHOD_LABEL)
print("Validation status: PASS")


Output: /content/drive/MyDrive/thesis_recsys/categories/herbal_supplements/outputs/stage1_candidate_pools/stage1_candidate_pool_export_manifest_herbal.json
Rows: queries 1968
Query-only winner: hybrid_dense_bm25 - Dense-BM25 Hybrid
Personalized winner: profile_sparse_qchs - Profile Sparse QCHS
Validation status: PASS
